# 04 — Scientific Knowledge Extraction

Extrae tecnologías, tareas, sensores, métricas, robots, cultivos, entornos y conceptos semánticos; además descubre términos no predefinidos.

> Notebook reproducible del Paper Semantic Bibliometric Toolkit v1.0.0. No elimina documentos ni convierte resultados exploratorios en conclusiones sin revisión manual.

## 1. Cargar corpus y taxonomía

In [ ]:

from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
current = Path.cwd().resolve()
for candidate in [current] + list(current.parents):
    if (candidate / "config.json").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró config.json.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core import (
    find_project_root, read_json, load_config, ensure_directories,
    load_scopus_csv, load_clean_corpus, normalize_spacing,
    normalize_identifier, normalize_title, normalize_keyword,
    split_semicolon, frequency_table, save_csv, save_json,
    markdown_table, combined_text
)

CONFIG = load_config(PROJECT_ROOT)
ensure_directories(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
THESAURUS_DIR = PROJECT_ROOT / "thesaurus"
VALIDATION_DIR = PROJECT_ROOT / "validation"

print("Project root:", PROJECT_ROOT)
print("Toolkit version:", CONFIG.get("toolkit_version"))
print("Configured corpus:", PROJECT_ROOT / CONFIG["input_csv"])

df = load_clean_corpus(PROJECT_ROOT)
from text_analysis import taxonomy_statistics, discover_ngrams, extract_model_candidates, extract_dataset_candidates
taxonomy = read_json(PROJECT_ROOT / "config" / "domain_taxonomy.json")


## 2. Aplicar taxonomía científica

In [ ]:

theme_stats, theme_docs = taxonomy_statistics(df, taxonomy)
save_csv(theme_stats, OUTPUT_DIR / "04_theme_statistics.csv")
save_csv(theme_docs, OUTPUT_DIR / "04_theme_document_matches.csv")
for family in theme_stats["Family"].unique():
    save_csv(theme_stats[theme_stats["Family"] == family].sort_values("Documents", ascending=False), OUTPUT_DIR / ("04_" + family.lower().replace("/", "_").replace(" ", "_") + "_statistics.csv"))
display(theme_stats)


## 3. Descubrimiento automático de n-gramas

In [ ]:

ngrams = discover_ngrams(df, min_df=2, max_features=3000)
save_csv(ngrams, OUTPUT_DIR / "04_discovered_ngrams.csv")
display(ngrams.head(100))


## 4. Candidatos de modelos y datasets

In [ ]:

model_candidates = extract_model_candidates(df)
dataset_candidates = extract_dataset_candidates(df)
save_csv(model_candidates, OUTPUT_DIR / "04_model_candidates.csv")
save_csv(dataset_candidates, OUTPUT_DIR / "04_dataset_candidates.csv")
display(model_candidates.head(50))
display(dataset_candidates.head(50))


## 5. Evidencia preliminar del research gap

In [ ]:

semantic = theme_stats[theme_stats["Family"] == "Semantic/Assistance"].sort_values(["Documents", "Concept"])
core_tasks = theme_stats[theme_stats["Family"] == "Task"].sort_values("Documents", ascending=False)
save_csv(semantic, OUTPUT_DIR / "04_gap_indicators.csv")
summary = {
    "Top tasks": core_tasks.head(10).to_dict("records"),
    "Semantic and assistance indicators": semantic.to_dict("records"),
    "Caution": "Low frequency is exploratory evidence and must be confirmed by targeted searches and manual reading."
}
save_json(summary, OUTPUT_DIR / "04_knowledge_extraction_summary.json")
text = (
    "Thematic dictionaries were applied to titles, abstracts, and keywords to identify technologies, tasks, sensors, metrics, "
    "robot types, crops, environments, and semantic-assistance concepts. TF-IDF n-gram discovery complemented the dictionaries "
    "to surface terminology that had not been specified in advance."
)
(REPORTS_DIR / "04_knowledge_extraction_methodology.txt").write_text(text, encoding="utf-8")
display(semantic)
